# Native categorical splits, unseen categories and SQL export

`BudgetClassifier` automatically uses `CategoricalInterleavedTreeClassifier` when given
a DataFrame with categorical or string columns. It orders categories by smoothed Newton
ratio with MDL regularization, separates missing values (`NaN`) from unseen test-time
categories, keeps near-unique timestamp strings ordered via a coverage guardrail, and
exports exact ANSI SQL (`IN (...)` / `NOT IN (...)` and `IS NULL`).

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

from credit_data import make_credit_data

X, y = make_credit_data(n=20_000, seed=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
print(f"{len(X_train):,} training rows, {len(X_test):,} test rows, "
      f"{y.mean():.1%} defaults, {X['income'].isna().mean():.0%} incomes missing")
rng = np.random.default_rng(42)
regions = np.array(["North", "South", "East", "West", "Central", "O'Connor"])
reg_idx = rng.integers(0, len(regions), size=len(X))
days = rng.integers(0, 336, size=len(X))
ts = [f"2026-{1 + int(d // 28):02d}-{1 + int(d % 28):02d}T{int(i % 86400):05d}Z"
      for i, d in enumerate(days)]
X_mix = X.copy()
X_mix["region"] = pd.Series(regions[reg_idx], dtype="object")
X_mix.loc[rng.random(len(X)) < 0.04, "region"] = None
X_mix["application_ts"] = pd.Series(ts, dtype="object")
logit_extra = 1.2 * np.isin(regions[reg_idx], ["North", "O'Connor"]) + 0.8 * (days > 168)
y_mix = pd.Series(
    (rng.random(len(X)) < 1 / (1 + np.exp(-(-2.1 + logit_extra + 0.5 * X["debt_ratio"])))),
    name="default",
).astype(int)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    X_mix, y_mix, test_size=0.3, random_state=0, stratify=y_mix)

14,000 training rows, 6,000 test rows, 10.3% defaults, 5% incomes missing


## Automatic categorical detection & bounded rule length

`BudgetClassifier` handles categoricals automatically; passing `max_subset_size=2` on
`CategoricalInterleavedTreeClassifier` keeps every categorical rule compact (at most 2
listed categories per split). Near-unique ISO-8601 timestamps (`application_ts`) trigger
the automatic coverage guardrail and stay chronologically ordered.

In [2]:
from bettertrees import BudgetClassifier, CategoricalInterleavedTreeClassifier

auto_model = BudgetClassifier(max_splits=8).fit(Xm_tr, ym_tr)
compact_cat = CategoricalInterleavedTreeClassifier(
    max_splits=8, max_delta_step=4.0, max_subset_size=2
).fit(Xm_tr, ym_tr)
print("Auto model type:", type(auto_model.model_).__name__)
print("Auto AUC:", round(roc_auc_score(ym_te, auto_model.predict_proba(Xm_te)[:, 1]), 4))
print("Compact (max_subset_size=2) AUC:",
      round(roc_auc_score(ym_te, compact_cat.predict_proba(Xm_te)[:, 1]), 4))
print("Guarded timestamp cols (ordered):", list(compact_cat._ord_vocabs.keys()))
print("\n" + compact_cat.explain())

Auto model type: CategoricalInterleavedTreeClassifier
Auto AUC: 0.684
Compact (max_subset_size=2) AUC: 0.684
Guarded timestamp cols (ordered): [8]

logit P(y = 1) = base -1.0515 + sum of 3 trees (8 cuts)
tree 1:
  +0.7135  if region in {'North', "O'Connor"}
  -0.0427  if region is missing
  -0.5235  if (region is present and region not in {'North', "O'Connor"})
tree 2:
  -0.4702  if application_ts <= 7000 and debt_ratio <= 0.4875
  -0.1710  if application_ts <= 7000 and debt_ratio > 0.4875
  +0.4178  if application_ts > 7000
tree 3:
  -0.2123  if (income <= 1.905e+04 or missing) and months_employed <= 47.5
  +0.3455  if (income <= 1.905e+04 or missing) and months_employed > 47.5 and age <= 51.5
  -0.1144  if (income <= 1.905e+04 or missing) and months_employed > 47.5 and age > 51.5
  +0.0683  if 1.905e+04 < income <= 4.115e+04
  -0.0536  if income > 4.115e+04


## Missing vs unseen categories and exact SQLite verification

An unseen category at test time (`"BrandNewRegion"`) routes to the majority side of each
split instead of inheriting missing-value (`None`) risk, and `to_sql()` reproduces
`decision_function` and `predict_proba` identically inside SQLite.

In [3]:
import math
import sqlite3

probe = Xm_te.iloc[:100].copy()
probe.iloc[0, probe.columns.get_loc("region")] = "BrandNewRegion"
probe.iloc[1, probe.columns.get_loc("region")] = None

sql = compact_cat.to_sql("applicants", precision=17)
print(sql[:700], "...")
with sqlite3.connect(":memory:") as con:
    con.create_function("EXP", 1, math.exp)
    probe.to_sql("applicants", con, index=False)
    out = pd.read_sql(sql, con)
max_diff = float(np.max(np.abs(out["p"].to_numpy() - compact_cat.predict_proba(probe)[:, 1])))
print(f"\nMax |SQL p - Python p| across 100 rows (incl. unseen & NULL): {max_diff:.2e}")

-- logit P(y = 1) = base + sum of 3 tree columns (8 cuts); p = 1 / (1 + EXP(-score))
WITH contributions AS (
  SELECT *,
    CASE  -- tree 1
      WHEN region IN ('North', 'O''Connor') THEN 0.71353296567669333
      WHEN region IS NULL THEN -0.042696957882111392
      ELSE -0.52347993218825883
    END AS t1_region,
    CASE  -- tree 2
      WHEN (application_ts <= '2026-07-02T09475Z' OR application_ts IS NULL) AND (debt_ratio <= 0.48749999701976776 OR debt_ratio IS NULL) THEN -0.47016922033537883
      WHEN (application_ts <= '2026-07-02T09475Z' OR application_ts IS NULL) AND debt_ratio > 0.48749999701976776 THEN -0.17103873159557847
      ELSE 0.41776393077810076
    END AS t2_application_t ...

Max |SQL p - Python p| across 100 rows (incl. unseen & NULL): 1.11e-16
